# MSTR TPC-DS Schema Import

Imports all `mstr_view` BigQuery views as logical tables into the MSTR project `mstr-tpc`.

**Approach A** — mstrio-py (`list_warehouse_tables` → `add_to_project`)  
**Approach B** — MSTR REST API using pre-built JSON files in `config/osi_import_json/`

References: [mstrio-py table_mgmt.py](https://github.com/MicroStrategy/mstrio-py/blob/master/code_snippets/table_mgmt.py)

## Imports & config

In [2]:
import json
import yaml
import os
import glob
import time
from mstrio.api import tables
import glob, json, os
from pprint import pprint

from mstrio.connection import Connection
from mstrio.modeling import SchemaManagement, SchemaUpdateType
from mstrio.modeling.schema.helpers import ObjectSubType, SchemaObjectReference
from mstrio.modeling.schema.table import (
    list_warehouse_tables,
    list_logical_tables,
    LogicalTable,
)


In [7]:
PROJECT_ID   = "35249BC4472353A88620DCAF854F304A"   # mstr-tpc project
FACTS_FOLDER_ID = "D4BB03B950C14CB0946D64FFC136FEF5"
DATASOURCE_ID="A23BBC514D336D5B4FCE919FE19661A3"
METRICS_FOLDER_ID = "E0CCB9CF22104A489CBE78D974AFD19E"
ATTR_FOLDER_ID = "6F55FB47F9974EABA18CB0C5FF46785C"

JSON_DIR      = "C:\\coding\\Python_environments\\OSI_Files"         # pre-built REST API payloads
TABLE_DIR     = ".\\mstr_tutorial\\table_def"
FACT_DIR      = ".\\mstr_tutorial\\fact_def"
ATTRIBUTE_DIR = ".\\mstr_tutorial\\attribute_def"
METRIC_DIR    = ".\\mstr_tutorial\\metric_def"

OSI_YAML = ".\\mstr_tutorial\\osi_import_schema.yaml"

os.chdir(JSON_DIR)

with open(OSI_YAML, "r", encoding="utf-8") as f:
    osi_model = yaml.safe_load(f)

## Connect to MSTR project

In [ ]:
user_conf="C:\\coding\\Python_environments\\mstr_robotics\\config\\user_d.json"
with open(user_conf, 'r') as openfile:
    user_d = json.load(openfile)
conn_params=user_d["conn_params"]
print(conn_params)
conn = Connection(
    base_url=conn_params["base_url"],
    username=conn_params["username"],
    password=conn_params["password"],
    project_id=PROJECT_ID
)
conn.headers["Content-type"] = "application/json"
print(f"Connected → project_id={conn.project_id}")


{'username': 'Administrator', 'password': '[REMOVED-PASSWORD]', 'base_url': 'http://217.154.213.84:8080/MicroStrategyLibrary/api'}
Connection to Strategy One Intelligence Server has been established.
Connected → project_id=35249BC4472353A88620DCAF854F304A


In [13]:
os.makedirs(TABLE_DIR, exist_ok=True)

datasets = osi_model["semantic_model"][0]["datasets"]
print(f"Found {len(datasets)} table(s) in OSI model\n")

for ds in datasets:
    table_name = ds["name"]
    source = ds.get("source", "")
    parts = source.split(".")
    # source format: <db>.<schema>.<table>
    namespace = parts[-2] if len(parts) >= 2 else ""
    namespace=""
    physical_name = parts[-1] if parts else table_name

    payload = {
        "information": {
            "name": table_name
        },
        "primaryDataSource": {
            "objectId": DATASOURCE_ID
        },
        "physicalTable": {
            "tableName": physical_name,
            "namespace": namespace
        }
    }

    out_path = os.path.join(TABLE_DIR, f"{table_name}.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

    print(f"  {table_name}.json")

print(f"\nDone — {len(datasets)} JSON files written to: {TABLE_DIR}")

Found 15 table(s) in OSI model

  order_fact.json
  order_detail.json
  lu_day.json
  lu_customer.json
  lu_item.json
  lu_subcateg.json
  lu_category.json
  rel_cat_item.json
  lu_country.json
  lu_region.json
  lu_pymt_type.json
  lu_employee.json
  lu_promotion.json
  lu_shipper.json
  lu_brand.json

Done — 15 JSON files written to: .\mstr_tutorial\table_def


## REST API: POST /api/model/tables using JSON files

In [14]:

json_files = sorted(glob.glob(os.path.join(TABLE_DIR, "*.json")))
print(f"Found {len(json_files)} files\n")

results = {"ok": [], "error": []}

for fp in json_files:
    tbl = os.path.splitext(os.path.basename(fp))[0]
    with open(fp, "r", encoding="utf-8") as f:
        d = json.load(f)
    try:
        resp = tables.post_table(connection=conn, data=d)
        if resp.status_code in (200, 201):
            print(f"  ✓  {tbl}  →  id={resp.json().get('id','?')}")
            results["ok"].append(tbl)
        else:
            print(f"  ✗  {tbl}  HTTP {resp.status_code}: {resp.text[:200]}")
            results["error"].append({"table": tbl, "error": resp.text[:200]})
    except Exception as e:
        print(f"  ✗  {tbl}: {e}")
        results["error"].append({"table": tbl, "error": str(e)})

print(f"\nDone — {len(results['ok'])} ok, {len(results['error'])} errors")

Found 15 files

  ✓  lu_brand  →  id=5F2366D2A4A04686A7CBEA1E581AB754
  ✓  lu_category  →  id=74519B3D61184D4EBC272BAD3A428523
  ✓  lu_country  →  id=3D61E4FDA95F4215B7B055BEE811D5D9
  ✓  lu_customer  →  id=5490C03ED915428F852EE7F0967BD19F
  ✓  lu_day  →  id=7F46C4DAC57F47A7A3E10FF240388A0E
  ✓  lu_employee  →  id=A07AC85AD6BC48448BB59E0207FAD055
  ✓  lu_item  →  id=869F81E0C28B4F07A8059F08586C50F5
  ✓  lu_promotion  →  id=14F40D593B894C61A5F952D9E4C7A69C
  ✓  lu_pymt_type  →  id=FF6200B090924427860FE7D705731618
  ✓  lu_region  →  id=17A09B72FB004FDBBED36CDA56E5AF0F
  ✓  lu_shipper  →  id=60D88D3BF609465E968192E1C09D63A0
  ✓  lu_subcateg  →  id=C02A9765A3F349AAB0F585E3C25A6B76
  ✓  order_detail  →  id=78EA50FAF85B41079A0B9269A370B820
  ✓  order_fact  →  id=768669B4EA5E4B95B8CE8F1919B0939D
  ✓  rel_cat_item  →  id=6866DE25E9CA4F7A88185172A9F35189

Done — 15 ok, 0 errors


## Reload schema after import

In [15]:
def reload_schema(conn, project_id: str):
    """
    Triggers a schema reload so imported tables are visible
    in reports and dashboards.
    """
    schema_mgr = SchemaManagement(connection=conn, project_id=project_id)
    task = schema_mgr.reload(update_types=[SchemaUpdateType.LOGICAL_SIZE])
    print(f"Schema reload triggered: {task}")
    return task


# Uncomment after import is complete
# reload_schema(conn, PROJECT_ID)
print("reload_schema() ready — run after import completes")

reload_schema() ready — run after import completes


## Facts — generate JSON files from OSI metrics

In [19]:
import re
import yaml

def get_table_id(conn, table_name: str, _cache: dict = {}) -> str:
    """
    Returns the MSTR objectId for a logical table by name.
    Results are cached after the first call so list_logical_tables
    is only called once per session.
    """
    if not _cache:
        tables = list_logical_tables(conn)
        _cache.update({t.name: t.id for t in tables})
        print(f"[get_table_id] cached {len(_cache)} logical tables")

    if table_name not in _cache:
        raise KeyError(
            f"Logical table '{table_name}' not found in project. "
            f"Available: {sorted(_cache)}"
        )
    return _cache[table_name]


def parse_ansi_expr(expr_str: str):
    """Parses 'AGG(table.column)' → (table_name, column_name)."""
    m = re.match(r"\w+\((\w+)\.(\w+)\)", expr_str.strip())
    if not m:
        raise ValueError(f"Cannot parse OSI expression: {expr_str!r}")
    return m.group(1), m.group(2)


def build_fact_payload(conn, metric: dict, folder_id: str) -> dict:
    ansi_expr = next(
        d["expression"] for d in metric["expression"]["dialects"]
        if d["dialect"] == "ANSI_SQL"
    )
    table_name, col_name = parse_ansi_expr(ansi_expr)
    tbl_id = get_table_id(conn, table_name)
    return {
        "information": {
            "name": metric["name"],
            "subType": "fact",
            "destinationFolderId": folder_id,
        },
        "dataType": {"type": "float", "precision": 8, "scale": -2147483648},
        "expressions": [{
            "expression": {"tokens": [{"value": col_name}]},
            "tables": [{
                "objectId": tbl_id,
                "subType": "logical_table",
                "name": table_name,
            }],
        }],
    }


def generate_fact_json_files(
    conn,
    osi_yaml_path: str,
    fact_dir: str,
    folder_id: str = "__FACTS_FOLDER_ID__",
) -> list:
    """
    Reads OSI YAML metrics and writes one JSON file per metric into fact_dir.
    Table IDs are resolved live from MSTR via get_table_id().
    Returns list of (metric_name, file_path) tuples.
    """
    with open(osi_yaml_path, "r", encoding="utf-8") as f:
        osi = yaml.safe_load(f)

    metrics = osi["semantic_model"][0]["metrics"]
    os.makedirs(fact_dir, exist_ok=True)
    results = []

    for metric in metrics:
        try:
            payload = build_fact_payload(conn, metric, folder_id)
            fp = os.path.join(fact_dir, f"{metric['name']}.json")
            with open(fp, "w", encoding="utf-8") as f:
                json.dump(payload, f, indent=2)
            results.append((metric["name"], fp))
        except Exception as e:
            print(f"  ✗  {metric['name']}  ({e})")
            continue
        tbl  = payload["expressions"][0]["tables"][0]["name"]
        col  = payload["expressions"][0]["expression"]["tokens"][0]["value"]
        tbl_id = payload["expressions"][0]["tables"][0]["objectId"]
        print(f"  ✓  {metric['name']}  ({tbl}.{col}  tbl_id={tbl_id})")

    print(f"\nGenerated {len(results)} fact JSON files in: {fact_dir}")
    return results


fact_files = generate_fact_json_files(conn, OSI_YAML, FACT_DIR)


[get_table_id] cached 15 logical tables
  ✓  gross_sales  (order_fact.gross_dollar_sales  tbl_id=768669B4EA5E4B95B8CE8F1919B0939D)
  ✓  net_sales  (order_fact.order_amt  tbl_id=768669B4EA5E4B95B8CE8F1919B0939D)
  ✓  total_cost  (order_fact.order_cost  tbl_id=768669B4EA5E4B95B8CE8F1919B0939D)
  ✓  gross_margin  (order_fact.order_amt  tbl_id=768669B4EA5E4B95B8CE8F1919B0939D)
  ✗  gross_margin_pct  (Cannot parse OSI expression: 'ROUND(100.0 * (SUM(order_fact.order_amt) - SUM(order_fact.order_cost)) / NULLIF(SUM(order_fact.order_amt), 0), 2)')
  ✓  units_sold  (order_fact.qty_sold  tbl_id=768669B4EA5E4B95B8CE8F1919B0939D)
  ✗  order_count  (Cannot parse OSI expression: 'COUNT(DISTINCT order_fact.order_id)')
  ✓  avg_order_value  (order_fact.gross_dollar_sales  tbl_id=768669B4EA5E4B95B8CE8F1919B0939D)
  ✗  customer_count  (Cannot parse OSI expression: 'COUNT(DISTINCT order_fact.customer_id)')
  ✓  customer_lifetime_value  (order_fact.gross_dollar_sales  tbl_id=768669B4EA5E4B95B8CE8F1919B093

## Facts — patch folder ID into JSON files, then create in MSTR

In [20]:
from mstrio.api import facts


def patch_fact_folder_id(fact_dir: str, folder_id: str) -> None:
    """Replaces __FACTS_FOLDER_ID__ in all fact JSON files with the real folder_id."""
    patched = 0
    for fp in sorted(glob.glob(os.path.join(fact_dir, "*.json"))):
        with open(fp, "r", encoding="utf-8") as f:
            body = json.load(f)
        body["information"]["destinationFolderId"] = folder_id
        with open(fp, "w", encoding="utf-8") as f:
            json.dump(body, f, indent=2)
        patched += 1
    print(f"Patched destinationFolderId={folder_id} in {patched} files")


def create_facts_from_json(conn, fact_dir: str) -> dict:
    """
    Loops through all JSON files in fact_dir and creates each as a MSTR fact
    via POST /api/model/facts (changeset managed automatically by mstrio-py).
    Returns dict with ok / error lists.
    """
    json_files = sorted(glob.glob(os.path.join(fact_dir, "*.json")))
    print(f"Found {len(json_files)} fact JSON files\n")

    results = {"ok": [], "error": []}

    for fp in json_files:
        fact_name = os.path.splitext(os.path.basename(fp))[0]
        with open(fp, "r", encoding="utf-8") as f:
            d = json.load(f)
        try:
            resp = facts.create_fact(connection=conn, body=d)
            if resp.status_code in (200, 201):
                fact_id = resp.json().get("id", "?")
                print(f"  ✓  {fact_name}  →  id={fact_id}")
                results["ok"].append({"name": fact_name, "id": fact_id})
            else:
                print(f"  ✗  {fact_name}  HTTP {resp.status_code}: {resp.text[:300]}")
                results["error"].append({"name": fact_name, "error": resp.text[:300]})
        except Exception as e:
            print(f"  ✗  {fact_name}: {e}")
            results["error"].append({"name": fact_name, "error": str(e)})

    print(f"\nDone — {len(results['ok'])} ok, {len(results['error'])} errors")
    return results


# 1. patch folder ID
patch_fact_folder_id(FACT_DIR, FACTS_FOLDER_ID)

# 2. create facts
fact_results = create_facts_from_json(conn, FACT_DIR)


Patched destinationFolderId=D4BB03B950C14CB0946D64FFC136FEF5 in 7 files
Found 7 fact JSON files

  ✓  avg_order_value  →  id=F34D8363541E4396973F35AF8DA1D8FD
  ✓  customer_lifetime_value  →  id=483A83305D8B4E369119A5BEB289453D
  ✓  gross_margin  →  id=992B67B4D13343908EB924679CCF602F
  ✓  gross_sales  →  id=F545D30100EB4F0A89B9F9974270D2FD
  ✓  net_sales  →  id=6136F5E724B24FD8A5DDEEDD66909EE0
  ✓  total_cost  →  id=56FC76E1B0B6425992026F003FF56EA3
  ✓  units_sold  →  id=7C8EB9BACE4341D792384CD2D1BC22DB

Done — 7 ok, 0 errors


## Metrics — generate JSON files from OSI metrics

In [23]:
from mstrio.modeling.schema.fact import list_facts

#METRIC_DIR = r"..\config\osi_create_metric"

# MSTR system-constant object ID for the Sum aggregation function
SUM_FUNCTION_ID = "8107C31BDD9911D3B98100C04F2233EA"

# metrics whose underlying measure is a count/quantity → integer data type
INTEGER_METRICS = {"store_quantity", "inventory_on_hand"}


def get_fact_id(conn, fact_name: str, _cache: dict = {}) -> str:
    """
    Returns the MSTR objectId for a fact by name.
    Results are cached after the first call so list_facts is only called once.
    """
    if not _cache:
        all_facts = list_facts(connection=conn)
        _cache.update({f.name: f.id for f in all_facts})
        print(f"[get_fact_id] cached {len(_cache)} facts")

    if fact_name not in _cache:
        raise KeyError(
            f"Fact '{fact_name}' not found in project. "
            f"Available: {sorted(_cache)}"
        )
    return _cache[fact_name]


def build_metric_payload(conn, metric_name: str, folder_id: str) -> dict:
    """
    Builds the REST API JSON body for a simple SUM(fact) metric.
    Expression tokens: Sum function → ( → fact object_reference → ) → end_of_text
    """
    fact_id = get_fact_id(conn, metric_name)
    dtype = "integer" if metric_name in INTEGER_METRICS else "float"
    precision, scale = (10, 0) if dtype == "integer" else (8, -2147483648)

    return {
        "information": {
            "name": metric_name,
            "subType": "metric",
            "destinationFolderId": folder_id,
        },
        "expression": {
            "tokens": [
                {
                    "value": "Sum",
                    "type": "function",
                    "target": {
                        "objectId": SUM_FUNCTION_ID,
                        "subType": "function",
                        "name": "Sum",
                    },
                },
                {"value": "(", "type": "character"},
                {
                    "value": metric_name,
                    "type": "object_reference",
                    "target": {
                        "objectId": fact_id,
                        "subType": "fact",
                        "name": metric_name,
                    },
                },
                {"value": ")", "type": "character"},
                {"value": "", "type": "end_of_text"},
            ]
        },
        "dataType": {"type": dtype, "precision": precision, "scale": scale},
    }


def generate_metric_json_files(
    conn,
    metrics: list,
    metric_dir: str,
    folder_id: str = "__METRICS_FOLDER_ID__",
) -> list:
    """
    Accepts the already-parsed OSI metrics list (no file I/O),
    resolves fact IDs from MSTR, and writes one JSON file per metric.
    Returns list of (metric_name, file_path) tuples.
    """
    os.makedirs(metric_dir, exist_ok=True)
    results = []

    for m in metrics:
        try:
            name = m["name"]
            payload = build_metric_payload(conn, name, folder_id)
            fp = os.path.join(metric_dir, f"{name}.json")
            with open(fp, "w", encoding="utf-8") as f:
                json.dump(payload, f, indent=2)
        except Exception as e:
            print(f"  ✗  {name}  ({e})")
            continue
        results.append((name, fp))
        fact_id = payload["expression"]["tokens"][2]["target"]["objectId"]
        dtype   = payload["dataType"]["type"]
        print(f"  ✓  {name}  (fact_id={fact_id}  dtype={dtype})")

    print(f"\nGenerated {len(results)} metric JSON files in: {metric_dir}")
    return results


# Parse OSI YAML once — reuse for both facts and metrics sections
with open(OSI_YAML, "r", encoding="utf-8") as f:
    osi_model = yaml.safe_load(f)

osi_metrics = osi_model["semantic_model"][0]["metrics"]
metric_files = generate_metric_json_files(conn, osi_metrics, METRIC_DIR)


`project_id` and `project_name` were not provided. Project from `connection` object is used instead.
[get_fact_id] cached 7 facts
  ✓  gross_sales  (fact_id=F545D30100EB4F0A89B9F9974270D2FD  dtype=float)
  ✓  net_sales  (fact_id=6136F5E724B24FD8A5DDEEDD66909EE0  dtype=float)
  ✓  total_cost  (fact_id=56FC76E1B0B6425992026F003FF56EA3  dtype=float)
  ✓  gross_margin  (fact_id=992B67B4D13343908EB924679CCF602F  dtype=float)
  ✗  gross_margin_pct  ("Fact 'gross_margin_pct' not found in project. Available: ['avg_order_value', 'customer_lifetime_value', 'gross_margin', 'gross_sales', 'net_sales', 'total_cost', 'units_sold']")
  ✓  units_sold  (fact_id=7C8EB9BACE4341D792384CD2D1BC22DB  dtype=float)
  ✗  order_count  ("Fact 'order_count' not found in project. Available: ['avg_order_value', 'customer_lifetime_value', 'gross_margin', 'gross_sales', 'net_sales', 'total_cost', 'units_sold']")
  ✓  avg_order_value  (fact_id=F34D8363541E4396973F35AF8DA1D8FD  dtype=float)
  ✗  customer_count  ("Fact '

## Metrics — patch folder ID into JSON files, then create in MSTR

In [24]:
from mstrio.api import metrics as metrics_api


def patch_metric_folder_id(metric_dir: str, folder_id: str) -> None:
    """Writes the real destinationFolderId into all metric JSON files."""
    patched = 0
    for fp in sorted(glob.glob(os.path.join(metric_dir, "*.json"))):
        with open(fp, "r", encoding="utf-8") as f:
            body = json.load(f)
        body["information"]["destinationFolderId"] = folder_id
        with open(fp, "w", encoding="utf-8") as f:
            json.dump(body, f, indent=2)
        patched += 1
    print(f"Patched destinationFolderId={folder_id} in {patched} metric files")


def create_metrics_from_json(conn, metric_dir: str) -> dict:
    """
    Loops through all JSON files in metric_dir and creates each as a MSTR
    metric via POST /api/model/metrics.
    Changeset lifecycle is managed automatically by mstrio-py.
    Returns dict with ok / error lists.
    """
    json_files = sorted(glob.glob(os.path.join(metric_dir, "*.json")))
    print(f"Found {len(json_files)} metric JSON files\n")

    results = {"ok": [], "error": []}

    for fp in json_files:
        metric_name = os.path.splitext(os.path.basename(fp))[0]
        with open(fp, "r", encoding="utf-8") as f:
            d = json.load(f)
        try:
            resp = metrics_api.create_metric(connection=conn, body=d)
            if resp.status_code in (200, 201):
                metric_id = resp.json().get("id", "?")
                print(f"  ✓  {metric_name}  →  id={metric_id}")
                results["ok"].append({"name": metric_name, "id": metric_id})
            else:
                print(f"  ✗  {metric_name}  HTTP {resp.status_code}: {resp.text[:300]}")
                results["error"].append({"name": metric_name, "error": resp.text[:300]})
        except Exception as e:
            print(f"  ✗  {metric_name}: {e}")
            results["error"].append({"name": metric_name, "error": str(e)})

    print(f"\nDone — {len(results['ok'])} ok, {len(results['error'])} errors")
    return results


# 1. patch real folder ID into all metric JSON files
patch_metric_folder_id(METRIC_DIR, METRICS_FOLDER_ID)

# 2. create metrics in MSTR
metric_results = create_metrics_from_json(conn, METRIC_DIR)


Patched destinationFolderId=E0CCB9CF22104A489CBE78D974AFD19E in 7 metric files
Found 7 metric JSON files

  ✓  avg_order_value  →  id=204F72EA0D494B0CA4C585ACB0824FDD
  ✓  customer_lifetime_value  →  id=B143380878DA49EDA7116C096E81E4E8
  ✓  gross_margin  →  id=6694649F4B054C1D9C222B099CE1F885
  ✓  gross_sales  →  id=39994518140945BE9D781BADB1BAA96D
  ✓  net_sales  →  id=1D9E8D69D8E64022A0032CBCADF48F33
  ✓  total_cost  →  id=EE08EC8B1E2A4B4EA4EBD3685F1A1E7E
  ✓  units_sold  →  id=03C34CCEE66F48319B917FB0C777C2DE

Done — 7 ok, 0 errors


## Attributes — generate JSON files from OSI datasets + relationships

In [25]:
from collections import defaultdict
from mstr_robotics.bq_connector import get_bq_client, get_bq_config
def discover_sk_relationships(bq_dataset: str = None) -> list[dict]:
    """
    Discovers FK→PK relationships between TPC-DS tables by inspecting all
    columns with the '_sk' suffix in the BigQuery INFORMATION_SCHEMA.

    Algorithm:
      1. Fetch every *_sk column from every table in bq_dataset.
      2. Strip the table prefix (all chars up to and including the first '_')
         to get the canonical key name.
             e.g.  "ss_item_sk"  →  "item_sk"
                   "i_item_sk"   →  "item_sk"
      3. The *owner* (PK / parent) of a canonical key is the table whose
         name + '_sk' equals that canonical key.
             e.g.  table='item', canonical='item_sk'  → owner ✓
      4. Every other table holding the same canonical key is a FK / child
         referencing the owner.

    Returns a list of relationship dicts ready for generate_parentchild_json_files:
        {
            "name":          "<from>_to_<to>",
            "from":          "<child_table>",   # FK side
            "to":            "<parent_table>",  # PK side
            "from_columns":  ["<fk_col>"],
            "to_columns":    ["<pk_col>"],
            "canonical_key": "<stripped_name>",
        }
    """
    bq_cfg = get_bq_config()
    project = bq_cfg["project"]
    if bq_dataset is None:
        bq_dataset = bq_cfg.get("dataset", bq_cfg.get("namespace", NAMESPACE))

    bq = get_bq_client()

    # ENDS_WITH avoids LIKE escape issues with '_' wildcards in BigQuery SQL
    query = f"""
        SELECT
            table_name,
            column_name
        FROM `{project}.{bq_dataset}.INFORMATION_SCHEMA.COLUMNS`
        WHERE ENDS_WITH(column_name, '_sk')
        ORDER BY table_name, column_name
    """
    rows = list(bq.query(query).result())
    print(f"Found {len(rows)} '_sk' columns in '{project}.{bq_dataset}'")

    # canonical_name → [(table, col), ...]
    canonical_map = defaultdict(list)
    for row in rows:
        col = row.column_name
        first_us = col.index("_")
        canonical = col[first_us + 1:]          # strip table prefix
        canonical_map[canonical].append((row.table_name, col))

    # owner: table whose name + '_sk' == canonical
    owner_map = {}   # canonical → (table, pk_col)
    for canonical, entries in canonical_map.items():
        for table, col in entries:
            if f"{table}_sk" == canonical:
                owner_map[canonical] = (table, col)
                break

    # build relationships
    relationships, seen = [], set()
    for canonical in sorted(canonical_map):
        if canonical not in owner_map:
            continue
        owner_table, owner_col = owner_map[canonical]
        for table, col in canonical_map[canonical]:
            if table == owner_table:
                continue
            key = (table, owner_table)
            if key in seen:           # keep first FK col per table pair
                continue
            seen.add(key)
            relationships.append({
                "name":          f"{table}_to_{owner_table}",
                "from":          table,
                "to":            owner_table,
                "from_columns":  [col],
                "to_columns":    [owner_col],
                "canonical_key": canonical,
            })
            print(f"  {table}.{col}  →  {owner_table}.{owner_col}  (key: {canonical})")

    print(f"\nDiscovered {len(relationships)} FK→PK relationships")
    return relationships


# ── run discovery ─────────────────────────────────────────────────────────────
bq_relationships = discover_sk_relationships(bq_dataset=NAMESPACE)


NameError: name 'NAMESPACE' is not defined

In [48]:
ATTR_DIR      = r"..\config\osi_create_attribute"
PC_DIR        = os.path.join(ATTR_DIR, "parentChild")
DTYPE_MAP_FILE = r"..\config\osi\db_enums\db_data_type_form_type.yaml"
os.makedirs(ATTR_DIR, exist_ok=True)
os.makedirs(PC_DIR, exist_ok=True)

# ── Load data-type mapping ────────────────────────────────────────────────────
with open(DTYPE_MAP_FILE, "r", encoding="utf-8") as _f:
    _dtype_cfg = yaml.safe_load(_f)

# Flat lookup keyed by UPPERCASE db type name
_TYPE_MAP         = {k.upper(): v for k, v in _dtype_cfg["type_map"].items()}
# Ordered list of (type_map_key, [suffixes]) — first match wins
_SUFFIX_HEURISTICS = list(_dtype_cfg["suffix_heuristics"].items())
_DTYPE_DEFAULT    = _dtype_cfg["default"]

print(f"Loaded {len(_TYPE_MAP)} db→mstr type mappings from {DTYPE_MAP_FILE}")

# osi_model was loaded in the metrics cell — reused here
datasets      = osi_model["semantic_model"]["datasets"]
osi_relations = osi_model["semantic_model"]["relationships"]


# ── helpers ───────────────────────────────────────────────────────────────────

def get_column_id(conn, table_name: str, column_name: str, _cache: dict = {}) -> str:
    """
    Returns the MSTR column objectId for a given table + column name.
    Calls LogicalTable.list_columns() once per table and caches the result.
    """
    if table_name not in _cache:
        table_id = get_table_id(conn, table_name)
        lt = LogicalTable(conn, id=table_id)
        cols = lt.list_columns()
        _cache[table_name] = {c.name: c.id for c in cols}
        print(f"[get_column_id] cached {len(_cache[table_name])} columns for '{table_name}'")

    col_map = _cache[table_name]
    if column_name not in col_map:
        raise KeyError(
            f"Column '{column_name}' not found in '{table_name}'. "
            f"Available: {sorted(col_map)}"
        )
    return col_map[column_name]


def _col_dtype(col_name: str, db_type: str | None = None) -> tuple[dict, str]:
    """
    Returns (dataType dict, displayFormat string) for an MSTR attribute form.

    Resolution order:
      1. db_type provided → case-insensitive lookup in _TYPE_MAP
      2. no db_type (or not found) → suffix heuristics from _SUFFIX_HEURISTICS
      3. fallback → _DTYPE_DEFAULT

    All type values come from config/osi/db_enums/db_data_type_form_type.yaml.
    """
    # 1 — direct db_type lookup
    if db_type:
        entry = _TYPE_MAP.get(db_type.upper())
        if entry:
            return (
                {"type": entry["mstr_type"], "precision": entry["precision"], "scale": entry["scale"]},
                entry["display_format"],
            )

    # 2 — suffix heuristics (checked in YAML declaration order)
    lc = col_name.lower()
    for type_key, suffixes in _SUFFIX_HEURISTICS:
        if any(lc.endswith(s) for s in suffixes):
            entry = _TYPE_MAP.get(type_key, _DTYPE_DEFAULT)
            return (
                {"type": entry["mstr_type"], "precision": entry["precision"], "scale": entry["scale"]},
                entry["display_format"],
            )

    # 3 — fallback
    return (
        {"type": _DTYPE_DEFAULT["mstr_type"], "precision": _DTYPE_DEFAULT["precision"], "scale": _DTYPE_DEFAULT["scale"]},
        _DTYPE_DEFAULT["display_format"],
    )


def build_attribute_payload(conn, dataset: dict, relations: list, folder_id: str) -> dict | None:
    """
    Builds the REST API JSON body for an MSTR attribute with full form coverage.

    ID form (category='ID'):
        • Expression 1: PK column on the lookup table itself
        • Expressions 2..N: FK columns on every fact/dim table that references
          this attribute (OSI relationships where this table is the 'to'/PK side).

    Descriptive forms (one per non-PK column in dataset["fields"]):
        • category='DESC' for the first descriptive column, 'NONE' for the rest
        • Single expression on the lookup table only
        • Data type resolved via _col_dtype() → db_data_type_form_type.yaml
    """
    table_name = dataset["name"]
    pk_cols    = dataset.get("primary_key", [])
    if not pk_cols:
        return None

    pk_col   = pk_cols[0]
    table_id = get_table_id(conn, table_name)
    col_id   = get_column_id(conn, table_name, pk_col)

    lookup_table_ref = {
        "objectId": table_id,
        "subType":  "logical_table",
        "name":     table_name,
    }

    # ── ID form: PK on lookup table + FK on every referencing table ──────────
    id_expressions = [
        {
            "expression": {"tokens": [
                {"value": pk_col, "type": "column_reference",
                 "target": {"objectId": col_id, "subType": "column", "name": pk_col}},
                {"value": "", "type": "end_of_text"},
            ]},
            "tables": [lookup_table_ref],
        }
    ]
    for rel in relations:
        if rel["to"] != table_name:
            continue
        fk_table = rel["from"]
        fk_cols  = rel.get("from_columns", [])
        if not fk_cols:
            continue
        fk_col = fk_cols[0]
        try:
            fk_table_id = get_table_id(conn, fk_table)
            fk_col_id   = get_column_id(conn, fk_table, fk_col)
            id_expressions.append({
                "expression": {"tokens": [
                    {"value": fk_col, "type": "column_reference",
                     "target": {"objectId": fk_col_id, "subType": "column", "name": fk_col}},
                    {"value": "", "type": "end_of_text"},
                ]},
                "tables": [{"objectId": fk_table_id, "subType": "logical_table", "name": fk_table}],
            })
        except KeyError as e:
            print(f"  ⚠  skipping FK expression {fk_table}.{fk_col} → {table_name}: {e}")

    forms = [
        {
            "name":          "ID",
            "alias":         pk_col,
            "category":      "ID",
            "displayFormat": "number",
            "dataType":      {"type": "integer", "precision": 8, "scale": 0},
            "expressions":   id_expressions,
            "lookupTable":   lookup_table_ref,
        }
    ]

    # ── Descriptive forms: one per non-PK column from OSI fields ─────────────
    non_pk_fields = [f for f in dataset.get("fields", []) if f["name"] != pk_col]
    desc_assigned = False

    for field in non_pk_fields:
        col_name = field["name"]
        # Use db_type from OSI model if present (field.get("type")), else suffix heuristic
        db_type = field.get("type")
        try:
            col_id_f = get_column_id(conn, table_name, col_name)
        except KeyError:
            print(f"  ⚠  '{col_name}' not in MSTR schema for '{table_name}' — skipping form")
            continue

        dtype, dfmt = _col_dtype(col_name, db_type=db_type)
        category    = "DESC" if not desc_assigned else "NONE"
        desc_assigned = True

        forms.append({
            "name":          col_name,
            "alias":         col_name,
            "category":      category,
            "displayFormat": dfmt,
            "dataType":      dtype,
            "expressions": [
                {
                    "expression": {"tokens": [
                        {"value": col_name, "type": "column_reference",
                         "target": {"objectId": col_id_f, "subType": "column", "name": col_name}},
                        {"value": "", "type": "end_of_text"},
                    ]},
                    "tables": [lookup_table_ref],
                }
            ],
            "lookupTable": lookup_table_ref,
        })

    browse_displays = [{"name": "ID"}]
    if non_pk_fields:
        browse_displays.append({"name": non_pk_fields[0]["name"]})

    return {
        "information": {
            "name":                table_name,
            "subType":             "attribute",
            "destinationFolderId": folder_id,
        },
        "forms":   forms,
        "keyForm": {"name": "ID"},
        "displays": {
            "reportDisplays": [{"name": "ID"}],
            "browseDisplays":  browse_displays,
        },
        "attributeLookupTable": lookup_table_ref,
    }


# ── generate attribute JSON files (one per dimension) ─────────────────────────

def generate_attribute_json_files(
    conn,
    datasets: list,
    relations: list,
    attr_dir: str,
    fact_tables: set,
    folder_id: str = "__ATTR_FOLDER_ID__",
) -> list:
    results = []
    for ds in datasets:
        name = ds["name"]
        if name in fact_tables:
            continue
        payload = build_attribute_payload(conn, ds, relations, folder_id)
        if payload is None:
            print(f"  ⚠  {name} — skipped (no primary_key in OSI)")
            continue
        fp = os.path.join(attr_dir, f"{name}.json")
        with open(fp, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2)
        results.append((name, fp))
        n_forms  = len(payload["forms"])
        n_id_exp = len(payload["forms"][0]["expressions"])
        print(f"  ✓  {name}  (forms={n_forms}  id_expressions={n_id_exp})")
    print(f"\nGenerated {len(results)} attribute JSON files in: {attr_dir}")
    return results


# ── generate parentChild JSON files (dim-to-dim relationships only) ───────────

def generate_parentchild_json_files(
    osi_relations: list,
    fact_tables: set,
    pc_dir: str,
) -> list:
    results = []
    for rel in osi_relations:
        from_tbl = rel["from"]
        to_tbl   = rel["to"]
        if from_tbl in fact_tables or to_tbl in fact_tables:
            continue
        payload = {
            "name":                   rel["name"],
            "parent_attribute_name":  to_tbl,
            "child_attribute_name":   from_tbl,
            "join_table_name":        from_tbl,
            "relationship_type":      "one_to_many",
        }
        fp = os.path.join(pc_dir, f"{rel['name']}.json")
        with open(fp, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2)
        results.append(rel["name"])
        print(f"  ✓  parent={to_tbl}  ←→  child={from_tbl}  (join via {from_tbl})")
    print(f"\nGenerated {len(results)} parentChild JSON files in: {pc_dir}")
    return results


# ── run ───────────────────────────────────────────────────────────────────────

print("=== Attribute JSON files ===")
attr_files = generate_attribute_json_files(
    conn, datasets, osi_relations, ATTR_DIR,
    fact_tables=set(tbl_cls["fact_tables"].keys()),
)

print("\n=== ParentChild JSON files ===")
pc_files = generate_parentchild_json_files(
    osi_relations, fact_tables=set(tbl_cls["fact_tables"].keys()), pc_dir=PC_DIR
)

Loaded 44 db→mstr type mappings from ..\config\osi\db_enums\db_data_type_form_type.yaml
=== Attribute JSON files ===
`project_id` and `project_name` were not provided. Project from `connection` object is used instead.
LogicalTable object named: 'date_dim' with ID: '2489F679BFAD4D6E93E2058AE193A0BD'
[get_column_id] cached 28 columns for 'date_dim'
`project_id` and `project_name` were not provided. Project from `connection` object is used instead.
LogicalTable object named: 'store_sales' with ID: '29B5688E62BE4C14A64779D33BFBECEA'
[get_column_id] cached 23 columns for 'store_sales'
`project_id` and `project_name` were not provided. Project from `connection` object is used instead.
LogicalTable object named: 'store_returns' with ID: 'E6A9E9B4094B46868BAD86A8B6B6B492'
[get_column_id] cached 20 columns for 'store_returns'
`project_id` and `project_name` were not provided. Project from `connection` object is used instead.
LogicalTable object named: 'catalog_sales' with ID: 'E06A1933316149669

## Attributes — discover destination folder ID

## Attributes — patch folder ID into JSON files, then create in MSTR

In [51]:
from mstrio.api import attributes as attr_api


def patch_attr_folder_id(attr_dir: str, folder_id: str) -> None:
    """Writes the real destinationFolderId into all attribute JSON files."""
    patched = 0
    for fp in sorted(glob.glob(os.path.join(attr_dir, "*.json"))):
        with open(fp, "r", encoding="utf-8") as f:
            body = json.load(f)
        body["information"]["destinationFolderId"] = folder_id
        with open(fp, "w", encoding="utf-8") as f:
            json.dump(body, f, indent=2)
        patched += 1
    print(f"Patched destinationFolderId={folder_id} in {patched} attribute files")


def create_attributes_from_json(conn, attr_dir: str) -> dict:
    """
    Loops through all JSON files directly in attr_dir (not the parentChild/
    sub-folder) and creates each as a MSTR attribute via POST /api/model/attributes.
    Changeset lifecycle is managed automatically by mstrio-py.
    Returns dict with ok / error / skipped lists.
    """
    json_files = sorted(glob.glob(os.path.join(attr_dir, "*.json")))
    print(f"Found {len(json_files)} attribute JSON files\n")

    results = {"ok": [], "error": [], "skipped": []}

    for fp in json_files:
        attr_name = os.path.splitext(os.path.basename(fp))[0]
        with open(fp, "r", encoding="utf-8") as f:
            d = json.load(f)
        n_exp = len(d.get("forms", [{}])[0].get("expressions", []))
        try:
            resp = attr_api.create_attribute(connection=conn, body=d)
            if resp.status_code in (200, 201):
                attr_id = resp.json().get("id", "?")
                print(f"  ✓  {attr_name}  →  id={attr_id}  (expressions={n_exp})")
                results["ok"].append({"name": attr_name, "id": attr_id})
            else:
                err_text = resp.text
                # 8004ccfc = duplicate name in folder → attribute already exists
                if "8004ccfc" in err_text:
                    print(f"  ~  {attr_name}  already exists — skipped")
                    results["skipped"].append(attr_name)
                else:
                    print(f"  ✗  {attr_name}  HTTP {resp.status_code}: {err_text[:300]}")
                    results["error"].append({"name": attr_name, "error": err_text[:300]})
        except Exception as e:
            err_str = str(e)
            if "8004ccfc" in err_str:
                print(f"  ~  {attr_name}  already exists — skipped")
                results["skipped"].append(attr_name)
            else:
                print(f"  ✗  {attr_name}: {err_str}")
                results["error"].append({"name": attr_name, "error": err_str})

    print(f"\nDone — {len(results['ok'])} created, {len(results['skipped'])} skipped (already exist), {len(results['error'])} errors")
    return results


# 1. patch real folder ID
print("=== Patching folder ID ===")
patch_attr_folder_id(ATTR_DIR, ATTR_FOLDER_ID)

# 2. create attributes in MSTR
print("\n=== Creating attributes in MSTR ===")
attr_results = create_attributes_from_json(conn, ATTR_DIR)

=== Patching folder ID ===
Patched destinationFolderId=6F55FB47F9974EABA18CB0C5FF46785C in 17 attribute files

=== Creating attributes in MSTR ===
Found 17 attribute JSON files

  ✓  call_center  →  id=01A6F6A51C754AD5A54D73D532ADE581  (expressions=2)
  ✓  catalog_page  →  id=858A5CE69A12441AB5BB50BA662D68BF  (expressions=2)
  ✓  customer  →  id=C6BCBD08ED1948719E3788399297378D  (expressions=2)
  ✓  customer_address  →  id=8302A71F2A7944D0AFCD02CE62FA6EBF  (expressions=2)
  ✓  customer_demographics  →  id=97B933DBC51E4871A5975723B0DD6812  (expressions=2)
  ✓  date_dim  →  id=D39E7944E32F43E19098B873A0E88798  (expressions=6)
  ✓  household_demographics  →  id=743BEA9B38CF40A4A150CC827C5E0EBA  (expressions=2)
  ✓  income_band  →  id=302EB3E5515F480D88194C863DFA1201  (expressions=2)
  ✓  item  →  id=104D91F5879144E7A4C1EA8F3E08E4A6  (expressions=6)
  ✓  promotion  →  id=3EF0DF49B4564BC79ED28FCA422D91BE  (expressions=4)
  ✓  reason  →  id=68B2EC256A5945C992FCEE44CA1D16D3  (expressions=2)
 

In [57]:

# ── Generate parent attribute JSON files from existing attribute form expressions ──
#
# Each existing dimension attribute JSON (e.g. store.json) has an ID form
# with expressions on multiple tables:
#   • The main lookup table  (e.g. store.s_store_sk)
#   • Fact-table FK columns  (e.g. store_sales.ss_store_sk,
#                                  store_returns.sr_store_sk)
#
# This cell extracts those fact-table expressions and produces:
#   1. One minimal attribute JSON per unique FK column →  ATTR_PARENT_DIR/
#   2. attribute_parent_relations.json  →  listing parent→child pairs so we can
#      wire up MSTR attribute hierarchies in the next step.

ATTR_PARENT_DIR      = os.path.join(os.path.dirname(ATTR_DIR), "osi_create_attribute_parent")
ATTR_PARENT_REL_FILE = os.path.join(os.path.dirname(ATTR_DIR), "osi_attribute_parent_relations.json")


def generate_parent_attributes_and_relations(
    attr_dir: str,
    out_attr_dir: str,
    out_rel_file: str,
    dest_folder_id: str,
) -> list:
    """
    For every dimension attribute JSON in *attr_dir*:
      • Find ID-form expressions that reference a table OTHER than the
        attribute's main attributeLookupTable.
      • For each such (table, column) pair write a minimal attribute JSON
        to *out_attr_dir* – these are the "parent" (degenerate) attributes
        that live inside the fact tables.
      • Record a relation dict  {parent_attribute, child_attribute, join_table}.

    Returns the full list of relation dicts.
    """
    os.makedirs(out_attr_dir, exist_ok=True)

    relations: list  = []
    seen_attrs: set  = set()          # avoid duplicate attribute files

    for attr_path in sorted(glob.glob(os.path.join(attr_dir, "*.json"))):
        with open(attr_path, "r", encoding="utf-8") as f:
            attr = json.load(f)

        main_attr_name  = attr["information"]["name"]
        main_lookup_id  = attr.get("attributeLookupTable", {}).get("objectId", "")

        # ── locate the ID form ────────────────────────────────────────────────
        id_form = next(
            (frm for frm in attr.get("forms", []) if frm.get("category") == "ID"),
            None,
        )
        if id_form is None:
            print(f"  ⚠  {main_attr_name}: no ID form found, skipping")
            continue

        # ── iterate over every expression in the ID form ──────────────────────
        for expr_item in id_form.get("expressions", []):
            for table in expr_item.get("tables", []):
                if table["objectId"] == main_lookup_id:
                    continue            # own lookup table – not a parent candidate

                # ── extract the column reference token ────────────────────────
                col_tok = next(
                    (t for t in expr_item["expression"]["tokens"]
                     if t.get("type") == "column_reference"),
                    None,
                )
                if col_tok is None:
                    continue

                col_name = col_tok["value"]   # e.g. "ss_store_sk"

                # ── record relation regardless of whether file already written ─
                relations.append({
                    "parent_attribute": col_name,
                    "child_attribute":  main_attr_name,
                    "join_table":       table["name"],
                })

                if col_name in seen_attrs:
                    continue            # attribute file already written
                seen_attrs.add(col_name)

                # ── build minimal parent-attribute payload ─────────────────────
                parent_attr = {
                    "information": {
                        "name":                col_name,
                        "subType":             "attribute",
                        "destinationFolderId": dest_folder_id,
                    },
                    "forms": [
                        {
                            "name":          "ID",
                            "alias":         col_name,
                            "category":      "ID",
                            "displayFormat": id_form["displayFormat"],
                            "dataType":      id_form["dataType"],
                            "expressions":   [expr_item],
                            "lookupTable": {
                                "objectId": table["objectId"],
                                "subType":  table["subType"],
                                "name":     table["name"],
                            },
                        }
                    ],
                    "keyForm":  {"name": "ID"},
                    "displays": {
                        "reportDisplays": [{"name": "ID"}],
                        "browseDisplays": [{"name": "ID"}],
                    },
                    "attributeLookupTable": {
                        "objectId": table["objectId"],
                        "subType":  table["subType"],
                        "name":     table["name"],
                    },
                }

                out_path = os.path.join(out_attr_dir, f"{col_name}.json")
                with open(out_path, "w", encoding="utf-8") as f:
                    json.dump(parent_attr, f, indent=2, ensure_ascii=False)

                print(f"  +  {col_name:35s}  (lookup: {table['name']})")

    # ── write relation file ───────────────────────────────────────────────────
    rel_payload = {"attribute_relations": relations}
    with open(out_rel_file, "w", encoding="utf-8") as f:
        json.dump(rel_payload, f, indent=2, ensure_ascii=False)

    print(f"\n  {len(seen_attrs)} parent attribute file(s) written → {out_attr_dir}")
    print(f"  {len(relations)} relation(s) written         → {out_rel_file}")
    return relations


# ── functions to push parent attributes + hierarchies into MSTR ───────────────

def create_parent_attributes_in_mstr(conn, out_attr_dir: str) -> dict:
    """Create all parent attribute JSON files in MSTR (reuses existing helper)."""
    return create_attributes_from_json(conn, out_attr_dir)


def create_parent_child_relations_in_mstr(conn, attr_relations: list) -> dict:
    """
    For each relation dict {parent_attribute, child_attribute, join_table}
    calls  parent_attr.add_child(child_ref, join_table_ref)  in MSTR.

    Both attributes must already exist in the project.
    """
    results = {"ok": [], "error": []}

    for rel in attr_relations:
        p_name = rel["parent_attribute"]
        c_name = rel["child_attribute"]
        t_name = rel["join_table"]

        try:
            parent_id   = get_attribute_id(conn, p_name)
            child_id    = get_attribute_id(conn, c_name)
            join_tbl_id = get_table_id(conn, t_name)

            parent_attr = Attribute(connection=conn, id=parent_id)

            child_ref = SchemaObjectReference(
                object_id=child_id,
                sub_type=ObjectSubType.ATTRIBUTE,
                name=c_name,
            )
            table_ref = SchemaObjectReference(
                object_id=join_tbl_id,
                sub_type=ObjectSubType.LOGICAL_TABLE,
                name=t_name,
            )
            parent_attr.add_child(
                child=child_ref,
                relationship_type=RelationshipType.ONE_TO_MANY,
                table=table_ref,
            )
            print(f"  ✓  {p_name}  ←[1:N]→  {c_name}  (join: {t_name})")
            results["ok"].append(f"{p_name}→{c_name}")

        except Exception as e:
            print(f"  ✗  {p_name}→{c_name}: {e}")
            results["error"].append({"relation": f"{p_name}→{c_name}", "error": str(e)})

    print(f"\nDone — {len(results['ok'])} ok, {len(results['error'])} errors")
    return results


# ── run: generate files ───────────────────────────────────────────────────────
print("=== Generating parent attribute files ===")
attr_parent_relations = generate_parent_attributes_and_relations(
    attr_dir      = ATTR_DIR,
    out_attr_dir  = ATTR_PARENT_DIR,
    out_rel_file  = ATTR_PARENT_REL_FILE,
    dest_folder_id= ATTR_FOLDER_ID,
)

# ── run: create parent attributes in MSTR ─────────────────────────────────────
print("\n=== Creating parent attributes in MSTR ===")
parent_attr_results = create_parent_attributes_in_mstr(conn, ATTR_PARENT_DIR)

# ── run: wire up parent→child hierarchies ─────────────────────────────────────
print("\n=== Setting parent→child attribute relationships ===")
parent_rel_results = create_parent_child_relations_in_mstr(conn, attr_parent_relations)


=== Generating parent attribute files ===
  +  cs_call_center_sk                    (lookup: catalog_sales)
  +  cs_catalog_page_sk                   (lookup: catalog_sales)
  +  ss_customer_sk                       (lookup: store_sales)
  +  ss_addr_sk                           (lookup: store_sales)
  +  ss_cdemo_sk                          (lookup: store_sales)
  +  ss_sold_date_sk                      (lookup: store_sales)
  +  sr_returned_date_sk                  (lookup: store_returns)
  +  cs_sold_date_sk                      (lookup: catalog_sales)
  +  ws_sold_date_sk                      (lookup: web_sales)
  +  inv_date_sk                          (lookup: inventory)
  +  ss_hdemo_sk                          (lookup: store_sales)
  +  hd_income_band_sk                    (lookup: household_demographics)
  +  ss_item_sk                           (lookup: store_sales)
  +  sr_item_sk                           (lookup: store_returns)
  +  cs_item_sk                           (lo

## Attributes — create parent/child relationships

In [53]:
import sys
sys.path.insert(0, r"..")

from mstrio.modeling.schema.attribute.attribute import Attribute, list_attributes
from mstrio.modeling.schema.attribute import RelationshipType
from mstrio.modeling.schema.helpers import SchemaObjectReference, ObjectSubType
from collections import defaultdict





# ── attribute relationship creation ──────────────────────────────────────────

# Fresh cache each cell run — avoids stale results if attributes were just created
_attr_id_cache: dict = {}

def get_attribute_id(conn, attr_name: str) -> str:
    """
    Returns the MSTR objectId for an attribute by name.
    Uses _attr_id_cache (module-level dict, reset on each cell execution).
    """
    if not _attr_id_cache:
        all_attrs = list_attributes(connection=conn)
        _attr_id_cache.update({a.name: a.id for a in all_attrs})
        print(f"[get_attribute_id] cached {len(_attr_id_cache)} attributes")

    if attr_name not in _attr_id_cache:
        raise KeyError(
            f"Attribute '{attr_name}' not found in project. "
            f"Available: {sorted(_attr_id_cache)}"
        )
    return _attr_id_cache[attr_name]


def create_attribute_relationships(conn, relations: list, fact_tables: set) -> dict:
    """
    Creates dimensional parent→child relationships in MSTR for a list of
    relationship dicts (as returned by discover_sk_relationships).

    Only processes dim→dim relationships (skips any involving fact_tables).
    Both attributes must already exist in MSTR.

    Returns dict with ok / error lists.
    """
    dim_rels = [
        r for r in relations
        if r["from"] not in fact_tables and r["to"] not in fact_tables
    ]
    print(f"Processing {len(dim_rels)} dim→dim relationships "
          f"(skipped {len(relations) - len(dim_rels)} fact-table refs)\n")

    results = {"ok": [], "error": []}

    for rel in dim_rels:
        parent_name = rel["to"]
        child_name  = rel["from"]
        join_table  = rel["from"]

        try:
            parent_id     = get_attribute_id(conn, parent_name)
            child_id      = get_attribute_id(conn, child_name)
            join_table_id = get_table_id(conn, join_table)

            parent_attr = Attribute(connection=conn, id=parent_id)

            child_ref = SchemaObjectReference(
                object_id=child_id,
                sub_type=ObjectSubType.ATTRIBUTE,
                name=child_name,
            )
            table_ref = SchemaObjectReference(
                object_id=join_table_id,
                sub_type=ObjectSubType.LOGICAL_TABLE,
                name=join_table,
            )

            parent_attr.add_child(
                child=child_ref,
                relationship_type=RelationshipType.ONE_TO_MANY,
                table=table_ref,
            )
            print(f"  ✓  {parent_name}  ←[1:N]→  {child_name}  (join: {join_table})")
            results["ok"].append(rel["name"])

        except Exception as e:
            print(f"  ✗  {rel['name']}: {e}")
            results["error"].append({"name": rel["name"], "error": str(e)})

    print(f"\nDone — {len(results['ok'])} ok, {len(results['error'])} errors")
    return results


# ── create relationships in MSTR ──────────────────────────────────────────────
rel_results = create_attribute_relationships(
    conn,
    relations=bq_relationships,
    fact_tables=set(tbl_cls["fact_tables"].keys()),
)

Processing 3 dim→dim relationships (skipped 27 fact-table refs)

`project_id` and `project_name` were not provided. Project from `connection` object is used instead.
[get_attribute_id] cached 34 attributes
Attribute object named: 'customer' with ID: 'C6BCBD08ED1948719E3788399297378D'
Error updating attribute C6BCBD08ED1948719E3788399297378D relationship.
I-Server Error 8004ccc7, Table (id '96FB756DE1534B78B42B2D128EF62502') cannot be used as the join table for a relationship involving attribute (id 'C6BCBD08ED1948719E3788399297378D').
Ticket ID: None
  ✗  web_page_to_customer: Table (id '96FB756DE1534B78B42B2D128EF62502') cannot be used as the join table for a relationship involving attribute (id 'C6BCBD08ED1948719E3788399297378D').; code: '8004ccc7', ticket_id: 'None'
Attribute object named: 'income_band' with ID: '302EB3E5515F480D88194C863DFA1201'
  ✓  income_band  ←[1:N]→  household_demographics  (join: household_demographics)
Attribute object named: 'item' with ID: '104D91F5879144E

In [ ]:
# ── Write bq_relationships back into osi_tpcds_1g.yaml ───────────────────────
#
# Naming convention: {from_table}_{from_column}_{to_table}
# Replaces the relationships: section in-place; all other sections are preserved.

OSI_YAML_PATH = r"..\config\osi_tpcds_1g.yaml"

def bq_rels_to_osi(bq_rels: list[dict]) -> list[dict]:
    """
    Converts bq_relationships dicts to OSI relationship format.
    Name = {from_table}_{from_columns[0]}_{to_table}
    """
    osi_rels = []
    for r in bq_rels:
        osi_rels.append({
            "name":         f"{r['from']}_{r['from_columns'][0]}_{r['to']}",
            "from":         r["from"],
            "to":           r["to"],
            "from_columns": r["from_columns"],
            "to_columns":   r["to_columns"],
        })
    return osi_rels


def update_osi_relationships(yaml_path: str, bq_rels: list[dict]) -> None:
    """
    Loads osi_tpcds_1g.yaml, replaces the relationships list with the
    OSI-formatted bq_relationships, and writes the file back.
    """
    with open(yaml_path, "r", encoding="utf-8") as f:
        model = yaml.safe_load(f)

    model["semantic_model"]["relationships"] = bq_rels_to_osi(bq_rels)

    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.dump(model, f, allow_unicode=True, sort_keys=False, default_flow_style=False)

    n = len(model["semantic_model"]["relationships"])
    print(f"Written {n} relationships to {yaml_path}")
    for r in model["semantic_model"]["relationships"]:
        print(f"  {r['name']}")


update_osi_relationships(OSI_YAML_PATH, bq_relationships)